#### Importing libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder

#### Pre-processing (improved as much as I could)

In [ ]:
DF = pd.read_csv('Housing.csv')
print(DF.head())
print(DF.isnull().sum()) # no missing values found

# Feature Engineering
Features = ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']
X = DF[Features]
y = DF['price']

# Converting categorical Features to numerical
CatFeatures = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']
NumFeatures = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']

# Scaling numerical Features
NumTrans = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# OneHot Encoding for categorical Features
CatTrans = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first'))  # drop='first' to avoid dummy variable trap
])

# Combining the Features again
Preprocessor = ColumnTransformer(
    transformers=[
        ('num', NumTrans, NumFeatures),
        ('cat', CatTrans, CatFeatures)
    ])

#### Creating a model pipeline and training the model

In [ ]:
ModelPipeline = Pipeline(steps=[
    ('preprocessor', Preprocessor),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression())
])
XTrain, XTest, YTrain, YTest = train_test_split(X, y, test_size=0.2, random_state=42)
ModelPipeline.fit(XTrain, YTrain)
YPred = ModelPipeline.predict(XTest)

In [ ]:
print(YPred)

#### Evaluating the model

In [ ]:
MSE = mean_squared_error(YTest, YPred)
R2 = r2_score(YTest, YPred)
print(f"Mean Squared Error: {MSE}")
print(f"R-squared: {R2}")
cvScores = cross_val_score(ModelPipeline, X, y, cv=5, scoring='r2')
print(f"Cross-validated R-squared: {np.mean(cvScores)}")

#### Predicting prices

In [ ]:
NewData = pd.DataFrame({
    'area': [1500, 2000],
    'bedrooms': [3, 4],
    'bathrooms': [2, 3],
    'stories': [2, 3],
    'mainroad': ['yes', 'yes'],
    'guestroom': ['no', 'no'],
    'basement': ['no', 'no'],
    'hotwaterheating': ['no', 'no'],
    'airconditioning': ['yes', 'yes'],
    'parking': [2, 3],
    'prefarea': ['yes', 'yes'],
    'furnishingstatus': ['furnished', 'furnished']
})

PredictedPrices = ModelPipeline.predict(NewData)
print(f"Predicted Prices: {PredictedPrices}")